# 🛒 Olist E-Commerce Sales — Data Cleaning Pipeline
### Notebook 1 of 2 &nbsp;|&nbsp; Analysis: `Olist_Sales_EDA.ipynb`

---

## Overview

This notebook prepares the raw Olist dataset for analysis.  
**Every cleaning decision is documented with the rationale** — not just what was done, but why.

### Dataset
The [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) contains real orders placed on Brazil's largest e-commerce marketplace between 2016 and 2018.

### Tables
| Table | Raw rows | Description |
|-------|----------|-------------|
| `orders` | 99,441 | One row per order — the **hub table** |
| `customers` | 99,441 | Customer city, state and unique identifier |
| `order_items` | 112,650 | Item-level: price, freight, product, seller |
| `payments` | 103,886 | Payment method and value per order |
| `reviews` | 99,224 | Customer review score and comment text |
| `products` | 32,951 | Product metadata and category (Portuguese) |
| `category` | 71 | Portuguese → English category name translation |

### Pipeline
```
1. Import raw CSVs
2. Profile every table  →  understand nulls, dtypes, duplicates
3. Clean each table individually
4. Validate join keys before merging
5. Join cleaned tables into two analytical dataframes
6. Export clean CSVs for EDA notebook and Power BI
```

> **Core principle:** Clean each table individually before joining.  
> Joining messy tables first makes data quality issues harder to trace and fix downstream.


## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime as dt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 2 · Load Raw Data

Loading all 7 CSV files in their original, unmodified state.  
No transformations are applied at this stage — we profile first, clean second.


In [ ]:
orders = pd.read_csv('/content/olist_orders_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
category = pd.read_csv('/content/product_category_name_translation.csv')


## 3 · Data Profiling

Running a profiling function on every table before touching any data.

**What profiling checks:**
- `shape` — row and column count
- `duplicates` — any fully identical rows
- `nulls` — missing values per column with percentage
- `dtypes` — current data type of each column

This audit establishes the baseline and informs every cleaning decision that follows.

**Summary of issues found:**

| Table | Key issues |
|-------|-----------|
| `orders` | 3 timestamp columns with nulls · all timestamps stored as strings |
| `customers` | Clean · city names inconsistently capitalised/accented |
| `order_items` | Clean |
| `payments` | Clean |
| `reviews` | 814 duplicate `review_id`s · 551 duplicate `order_id`s · 58.7% null comment text |
| `products` | 610 null category names (1.9%) |
| `category` | Clean |


In [ ]:
def profile_table(df, name):
  print(f"{'='*50}")
  print(f"TABLE: {name}")
  print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
  print(f"Duplicates: {df.duplicated().sum():,}")
  print(f"\n Nulls per column: ")
  nulls = df.isnull().sum()
  for col, n in nulls[nulls>0].items():
    print(f"   {col}: {n:,} ({n/len(df)*100:.1f}%)")
  if nulls.sum() == 0:
    print("  None - clean!")
  print(f"\n Dtypes:")
  for col, dtype in df.dtypes.items():
    print(f"  {col}: {dtype}")
  print()

# Calling profiling function for every table
for df, name in [(orders,'Orders'), (customers,'Customers'), (order_items,'Order Items'), (payments,'Payments'), (reviews,'Reviews'), (products,"Products"), (category,'Category')]:
  profile_table(df, name)


TABLE: Orders
Shape: 99,441 rows x 8 columns
Duplicates: 0

 Nulls per column: 
   order_approved_at: 160 (0.2%)
   order_delivered_carrier_date: 1,783 (1.8%)
   order_delivered_customer_date: 2,965 (3.0%)

 Dtypes:
  order_id: object
  customer_id: object
  order_status: object
  order_purchase_timestamp: object
  order_approved_at: object
  order_delivered_carrier_date: object
  order_delivered_customer_date: object
  order_estimated_delivery_date: object

TABLE: Customers
Shape: 99,441 rows x 5 columns
Duplicates: 0

 Nulls per column: 
  None - clean!

 Dtypes:
  customer_id: object
  customer_unique_id: object
  customer_zip_code_prefix: int64
  customer_city: object
  customer_state: object

TABLE: Order Items
Shape: 112,650 rows x 7 columns
Duplicates: 0

 Nulls per column: 
  None - clean!

 Dtypes:
  order_id: object
  order_item_id: int64
  product_id: object
  seller_id: object
  shipping_limit_date: object
  price: float64
  freight_value: float64

TABLE: Payments
Shape: 10

## 4 · Clean Individual Tables

---

### 4.1 · Orders Table

**Issues identified and decisions made:**

| Issue | Column | Count | Decision | Rationale |
|-------|--------|-------|----------|-----------|
| Non-delivered orders | `order_status` | 2,963 | **Drop** | Revenue analysis requires complete transactions only |
| Null delivery dates | `order_delivered_customer_date` | 8 | **Drop** | Cannot compute delivery metrics without actual delivery date |
| Null approval timestamps | `order_approved_at` | 14 | **Fill** with `order_purchase_timestamp` | Delivered orders with no approval record — purchase time is the closest proxy |
| Null carrier dates | `order_delivered_carrier_date` | 1 | **Drop** | Single row — too few to impute reliably |
| String timestamps | All 5 timestamp columns | All | **Convert** to `datetime` | Required for date arithmetic and filtering |

**Derived columns added:**

| Column | Formula | Purpose |
|--------|---------|---------|
| `delivery_days` | `delivered_date − purchase_timestamp` | Operational delivery time metric |
| `is_late` | `delivered_date > estimated_date` | Late delivery flag |
| `purchase_month` | `.dt.to_period('M')` | Monthly grouping for revenue trends |
| `purchase_year` | `.dt.year` | Annual grouping |
| `purchase_day` | `.dt.day_name()` | Day-of-week patterns |
| `purchase_hour` | `.dt.hour` | Hour-of-day patterns |

> **Zero-null assertion:** The function asserts `df.isnull().sum().sum() == 0` at the end.  
> If any nulls remain after cleaning, the function raises an error immediately  
> rather than silently passing corrupted data downstream.


In [ ]:
def clean_orders(df):
  df = df.copy()

  # Convert all timestamp columns to datetime
  ts_cols = ['order_purchase_timestamp','order_approved_at', 'order_delivered_carrier_date','order_delivered_customer_date', 'order_estimated_delivery_date']
  for col in ts_cols:
    df[col] = pd.to_datetime(df[col])

  # Keep only delivered orders for revenue analysis
  print(f"Order Statuses: {df.order_status.value_counts().to_dict()}")
  df = df[df.order_status == 'delivered'].copy()
  print(f"After filtering to delivered: {len(df):,} rows")

  # Drop rows missing actual delivery date
  before = len(df)
  df = df.dropna(subset = ['order_delivered_customer_date'])
  print(f"Dropped {before - len(df):,} rows with null delivery date")

  # Fill null order_approved_at with purchase timestamp (14 rows)
  # These are delivered orders with no approval record — purchase time is best proxy
  df['order_approved_at'] = df['order_approved_at'].fillna(df['order_purchase_timestamp'])

  # Drop the 1 row with null carrier date — too little data to impute reliably
  df = df.dropna(subset= ['order_delivered_carrier_date'])

  # Create useful derived columns
  df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
  df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date'])
  df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period("M")
  df['purchase_year'] = df['order_purchase_timestamp'].dt.year
  df['purchase_day'] = df['order_purchase_timestamp'].dt.day_name()
  df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

  # Final validation

  assert df.order_id.duplicated().sum() == 0, "Duplicate order ids found!"
  assert df.isnull().sum().sum() == 0, f"Unexpected nulls remain:\n {df.isnull.sum()[df.isnull().sum() > 0]}"

  return df

orders_clean = clean_orders(orders)
print(f"\norders_clean: {orders_clean.shape}")



Order Statuses: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
After filtering to delivered: 96,478 rows
Dropped 8 rows with null delivery date

orders_clean: (96469, 14)


---

### 4.2 · Order Items Table

**Important — two outputs from this function:**

| Output | Grain | Used for |
|--------|-------|---------|
| `items_clean` | One row per item | Product-level analysis, category revenue |
| `orders_revenue` | One row per order (aggregated) | Revenue totals joined to `master_orders` |

**Revenue columns:**

| Column | Meaning |
|--------|---------|
| `revenue` | Product price only (no shipping) |
| `freight` | Shipping cost only |
| `total_revenue` | `revenue + freight` — what the customer actually paid |

> **`order_item_id` is NOT a unique key** across the table.  
> It is the sequence number of items within a single order (1, 2, 3...).  
> An order with 3 products has `order_item_id` values 1, 2, 3 — all for the same `order_id`.  
> Never use `order_item_id` as a join key.


In [ ]:
def clean_order_items(df):
  df = df.copy()

  # Basic validation
  print(f"Nulls: {df.isnull().sum().sum()}")
  print(f"Zero Price Orders: {(df['price'] <=0).sum()}")

  # Remove zero price rows
  df = df[df['price']>0].copy()

  # Create total value per item
  df['item_total'] = df['price'] + df['freight_value']

  # Aggregate to Order level
  order_level = df.groupby('order_id').agg(
      revenue = ('price','sum'),
      freight = ('freight_value', 'sum'),
      item_count = ('order_item_id', 'count'),
  ).reset_index()

  order_level['total_revenue'] = order_level['revenue'] + order_level['freight']

  print(f"\n order_items_clean(order_level): {order_level.shape}")
  print(f"Revenue Range: R${order_level.revenue.min():.2f} - R${order_level.revenue.max():,.2f}")

  return df, order_level

items_clean, orders_revenue = clean_order_items(order_items)



Nulls: 0
Zero Price Orders: 0

 order_items_clean(order_level): (98666, 5)
Revenue Range: R$0.85 - R$13,440.00


---

### 4.3 · Customers Table

**Critical distinction — two customer ID columns:**

| Column | Behaviour | Use for |
|--------|-----------|---------|
| `customer_id` | **Changes per order** — the same person gets a new ID for each purchase | Join key only |
| `customer_unique_id` | Stable — identifies the actual person across all orders | RFM, repeat rate, cohort analysis |

> ⚠️ **Using `customer_id` for any customer-level analysis will make every customer  
> appear as a one-time buyer.** Always use `customer_unique_id` for customer behaviour analysis.

**City standardisation applied:**
```python
.str.strip()               # remove leading/trailing whitespace
.str.lower()               # normalise case
.str.normalize('NFKD')    # decompose accented characters
.str.encode('ascii', errors='ignore').str.decode('utf-8')  # remove diacritics
```
This ensures `"São Paulo"`, `"sao paulo"`, and `"SAO PAULO"` are all treated as the same city.

**State validation:** Asserts exactly ≤ 27 unique states (Brazil has 26 states + 1 federal district).  
All 27 states confirmed present in the dataset.


In [ ]:
from numpy._core.defchararray import strip
def clean_customers(df):
  df = df.copy()

  # Standardize city names
  df['customer_city'] = (df['customer_city']
                         .str.strip()
                         .str.lower()
                         .str.normalize('NFKD')
                         .str.encode('ascii', errors='ignore')
                         .str.decode('utf-8'))

  # Validate state codes
  valid_states = df['customer_state'].unique()
  print(f"Unique states: {len(valid_states)} - {sorted(valid_states)}")
  assert len(valid_states) <= 27, "Unexpected number of states"

  print(f"\nUnique customers (uniue_id): {df['customer_unique_id'].nunique():,}")
  print(f"Unique customers (id):         {df['customer_id'].nunique():,}")
  print("Note: these differ because one person can have multiple orders")
  print(f"\ncustomers_clean: {df.shape}")

  return df

customers_clean = clean_customers(customers)




Unique states: 27 - ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

Unique customers (uniue_id): 96,096
Unique customers (id):         99,441
Note: these differ because one person can have multiple orders

customers_clean: (99441, 5)


---

### 4.4 · Reviews Table — Investigation Required

The reviews table required investigation before cleaning could begin.  
Two separate data quality issues were discovered, each requiring a different fix.

**Investigation step 1 — are `review_id`s unique?**


In [ ]:
# Check 1 — are review_ids unique?
print(f"Duplicate review_id count: {reviews['review_id'].duplicated().sum()}")

# Investigating further
# Step 1: how many times does each review_id appear?
review_id_counts = reviews['review_id'].value_counts()
print(review_id_counts.value_counts())

# Step 2: Look at duplicated review_id example
dup_review_id = reviews['review_id'].value_counts()[reviews['review_id'].value_counts() > 1].index[0]
example = reviews[reviews['review_id'] == dup_review_id]
print(example[['order_id','review_id','review_score',
               'review_comment_message','review_creation_date']].to_string())

Duplicate review_id count: 814
count
1    97621
2      764
3       25
Name: count, dtype: int64
                               order_id                         review_id  review_score                                                              review_comment_message review_creation_date
58393  cfdfd7862e532c4ec1ed1c6a1b56d320  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00
92768  a2ac6dad85cf8af5b0afb510a240fe8c  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00
95402  73a8ed1078e1d475d2b1fb7216a52e1a  4548534449b1f572e357211b90724f1b             1  quero devolver o produto comprei um veio outro;;;\r\nquero meu dinheiro de volta.   2018-08-18 00:00:00


**Investigation step 2 — are `order_id`s unique?**


In [ ]:
# Check 2 — how many order_ids appear more than once?
# Step 1 — check if order_id is unique in the reviews table
print(reviews['order_id'].duplicated().sum())
# If this prints anything > 0, there are duplicate order_ids

551


**Investigation step 3 — how many orders have multiple reviews?**


In [ ]:
# Step 2 — how many orders have multiple reviews?
review_counts = reviews.groupby('order_id')['review_id'].count()
print(review_counts.value_counts())

review_id
1    98126
2      543
3        4
Name: count, dtype: int64


**Investigation step 4 — inspect an actual duplicate to understand the cause**


In [ ]:
# Look at the actual duplicate rows side by side
dup_order_ids = reviews[reviews['order_id'].duplicated(keep=False)]['order_id'].unique()

# Pick one example and inspect it fully
example = reviews[reviews['order_id'] == dup_order_ids[0]]
print(example[['order_id','review_id','review_score',
               'review_comment_message','review_creation_date']].to_string())

                              order_id                         review_id  review_score review_comment_message review_creation_date
30    cf73e2cb1f4a9480ed70c154da3d954a  540e7bbb2d06cfb7f85f3a88ba7ac97f             5                    NaN  2018-01-18 00:00:00
3109  cf73e2cb1f4a9480ed70c154da3d954a  aa193e76d35950c4ae988237bb36ed2b             5                    NaN  2018-01-18 00:00:00


**Investigation findings — two distinct issues identified:**

#### Issue 1 · Duplicated `review_id` across different orders (data entry error)

- 764 `review_id`s appear **twice**, 25 appear **three times**
- Each duplicated `review_id` is assigned to **completely different `order_id`s** with identical text and score
- This is a source data error — the same review record was copy-pasted to multiple orders during data collection
- **Fix:** Drop all rows where `review_id` appears more than once  
- **Rationale:** Cannot determine which order the review genuinely belongs to

#### Issue 2 · Same order submitted review form multiple times (user behaviour)

- 543 orders have 2 review submissions, 4 orders have 3 submissions
- Same `order_id`, **different** `review_id`s, same date — customer clicked submit multiple times
- **Fix:** Keep only the most recent submission per `order_id`  
- **Rationale:** Latest submission represents the customer's final recorded response

#### Two output versions

| Output | Rows | Purpose |
|--------|------|---------|
| `reviews_all` | 97,530 | Score analysis — retains nulls in comment column |
| `reviews_text_clean` | 36,848 | Sentiment analysis — only rows with comment text |

> **Selection bias note:** Customers who leave text reviews may rate differently  
> from those who only leave a score. Always keep separate versions —  
> dropping nulls before score analysis would introduce selection bias.


In [ ]:
def clean_reviews(df):
  df = df.copy()

  # COnvert timestamps
  df['review_creation_date'] = pd.to_datetime(df['review_creation_date'])
  df['review_answer_timestamp'] = pd.to_datetime(df['review_answer_timestamp'])

  # Validate review_score
  assert df['review_score'].between(1,5).all(), "Scored outside 1-5 range!"

  # Issue 1 — drop rows where review_id is duplicated across different orders
  # Same review_id + same text assigned to multiple orders = data entry error
  # Cannot determine which order the review truly belongs to — safest to drop
  before = len(df)
  dup_review_ids = df['review_id'].value_counts()
  dup_review_ids = dup_review_ids[dup_review_ids > 1].index
  df= df[~df['review_id'].isin(dup_review_ids)].copy()
  print(f"Dropped {before - len(df):,} rows with duplicate review_ids")

  # Issue 2 — after above, handle any remaining duplicate order_ids
  # (same order submitted review form twice — keep latest)
  before = len(df)
  df = (df.sort_values('review_creation_date', ascending=False).drop_duplicates(subset='order_id', keep='first').reset_index(drop=True))
  print(f"Dropped {before- len(df):,} duplicate review submissions(kept latest)")

  # Verify both issues are resolved
  assert df['review_id'].duplicated().sum() == 0, "review_id still has duplicates!"
  assert df['order_id'].duplicated().sum() == 0, "order_id still has dupicates!"

  # Split into two versions
  reviews_all = df.copy()
  reviews_text = df.dropna(subset=['review_comment_message'])
  reviews_text = reviews_text[reviews_text['review_comment_message'].str.len() > 10].copy()

  print(f"\nreviews_all: {len(reviews_all):,} rows")
  print(f"reviews_text: {len(reviews_text):,} rows")
  return reviews_all, reviews_text

reviews_clean, reviews_text_clean = clean_reviews(reviews)




Dropped 1,603 rows with duplicate review_ids
Dropped 91 duplicate review submissions(kept latest)

reviews_all: 97,530 rows
reviews_text: 36,848 rows


---

### 4.5 · Products & Payments Tables

**Products:**
- 610 null `product_category_name` values (1.9%) → filled with `'unknown'` rather than dropped  
  *(these products have real sales — dropping them would silently remove revenue from the analysis)*
- Joined English translation from `product_category_name_translation.csv`
- Only `product_id` and `category_english` retained — all other columns are unused in analysis

**Payments:**
- Removed zero-value payment rows
- Fixed zero installments → replaced with 1 (a payment with 0 installments is not meaningful)
- Aggregated to order level:
  - Primary payment type: `payment_sequential == 1` (first payment method per order)
  - Total value: `sum(payment_value)` per `order_id`

**Resulting columns:** `order_id · payment_type · payment_value`


In [ ]:
def clean_products(df, translation_df):
  df= df.copy()
  # Fill null category with "unknown"
  df['product_category_name'] = df['product_category_name'].fillna('unknown')

  # Join English translation using the table product_category_name_translation
  df= df.merge(translation_df, on='product_category_name', how= 'left')
  df['category_english'] = df['product_category_name_english'].fillna('unknown')

  # Keep only what we need
  return df[['product_id','category_english']]

def clean_payments(df):
  df = df.copy()
  df = df[df['payment_value'] > 0].copy()
  # Fix zero installments
  df['payment_installments'] = df['payment_installments'].replace(0,1)

  # Get primary payment per order (sequential == 1)
  primary = df[df['payment_sequential'] == 1][['order_id','payment_type']]

  # Get total payemnt value per order
  total = df.groupby('order_id')['payment_value'].sum().reset_index()
  return primary.merge(total, on='order_id')

products_clean = clean_products(products, category)
payments_clean = clean_payments(payments)
print("All tables cleaned!!")



All tables cleaned!!


In [ ]:
payments_clean.columns



Index(['order_id', 'payment_type', 'payment_value'], dtype='object')

## 5 · Join Strategy & Key Validation

### Join strategy

`orders` is the **hub table** — every other table joins to it.  
The final `master_orders` dataframe maintains **order-level grain: one row = one delivered order**.

| Table joined | Type | Reason |
|-------------|------|--------|
| `orders_revenue` (from `order_items`) | **INNER** | Every order must have items — orders without items are data errors |
| `customers_clean` | **INNER** | Every order must have a customer by definition |
| `reviews_clean` | **LEFT** | 1,696 orders have no review — expected, not missing data |
| `payments_clean` | **LEFT** | 79 orders have no payment record — retain with null payment fields |

### Why validate before joining?

A dirty join key causes silent data corruption:
- **Null keys** → unexpected NaN rows after join
- **Duplicate keys** → row multiplication (fan-out) that inflates counts and revenue totals
- **Key mismatch** → unexpected row loss after inner joins

Running validation before every join catches these issues before they silently corrupt the data.


In [ ]:
def validate_join_keys(left_df, right_df, key, left_name, right_name):
  left_nulls = left_df[key].isnull().sum()
  right_nulls = right_df[key].isnull().sum()
  left_dupes = left_df[key].duplicated().sum()
  right_dupes = right_df[key].duplicated().sum()

  print(f"\nJoin Key validation: {key}")
  print(f"{left_name} - nulls: {left_nulls}, dupes: {left_dupes}")
  print(f"{right_name} - nulls: {right_nulls}, dupes: {right_dupes}")

  # Overlap Check
  left_keys = set(left_df[key].dropna())
  right_keys = set(right_df[key].dropna())
  in_left_only = len(left_keys - right_keys)
  in_right_only = len(right_keys - left_keys)
  in_both = len(left_keys & right_keys)

  print(f" Keys in both: {in_both}")
  print(f" In {left_name} only: {in_left_only:,} (will be nulls after left join)")
  print(f" In {right_name} only: {in_right_only:,} (will be dropped in left join)")

# Run before every join
validate_join_keys(orders_clean, orders_revenue, 'order_id', 'orders', 'order_items')
validate_join_keys(orders_clean, customers_clean, 'customer_id', 'orders', 'customers')
validate_join_keys(orders_clean, reviews_clean, 'order_id', 'orders', 'reviews')
validate_join_keys(orders_clean, payments_clean, 'order_id', 'orders', 'payments')





Join Key validation: order_id
orders - nulls: 0, dupes: 0
order_items - nulls: 0, dupes: 0
 Keys in both: 96469
 In orders only: 0 (will be nulls after left join)
 In order_items only: 2,197 (will be dropped in left join)

Join Key validation: customer_id
orders - nulls: 0, dupes: 0
customers - nulls: 0, dupes: 0
 Keys in both: 96469
 In orders only: 0 (will be nulls after left join)
 In customers only: 2,972 (will be dropped in left join)

Join Key validation: order_id
orders - nulls: 0, dupes: 0
reviews - nulls: 0, dupes: 0
 Keys in both: 94773
 In orders only: 1,696 (will be nulls after left join)
 In reviews only: 2,757 (will be dropped in left join)

Join Key validation: order_id
orders - nulls: 0, dupes: 0
payments - nulls: 0, dupes: 0
 Keys in both: 96390
 In orders only: 79 (will be nulls after left join)
 In payments only: 2,967 (will be dropped in left join)


## 6 · Build Master Analytical Dataframes

---

### 6.1 · `master_orders` — Order-level grain

Joins all cleaned tables through `orders` as the hub.  
**One row = one delivered order.**

Used for: revenue analysis · customer analysis · RFM segmentation · geographic analysis · review analysis

**Join sequence:**
```
orders_clean
  ← orders_revenue      (INNER — revenue per order)
  ← customers_clean     (INNER — city, state, customer_unique_id)
  ← reviews_clean       (LEFT  — review score, ~1,696 orders have no review)
  ← payments_clean      (LEFT  — payment type and value, 79 orders have no payment record)
```


In [ ]:
def build_master_orders(orders_clean, orders_revenue, customers_clean, reviews_clean, payments_clean):
  df = orders_clean.copy()
  n0 = len(df)
  print(f"Start: {n0:,} rows")

  # Join revenue (inner join - only keep orders that have items)
  df = df.merge(orders_revenue, how='inner', on='order_id')
  print(f"After joining revenue: {len(df):,} rows (dropped {n0-len(df):,} orders with no items)")

  # Join customers (inner join - need customers info)
  df = df.merge(customers_clean[['customer_id','customer_unique_id','customer_city','customer_state' ]], how='inner', on='customer_id')
  print(f"After joining customers: {len(df):,} rows")

  # Join reviews (Left join - not every order has a review)
  df = df.merge(reviews_clean[['order_id', 'review_score']], how='left', on='order_id')
  print(f"After joining reviews: {len(df):,} rows")
  print(f"  Orders with reviews: {df['review_score'].notna().sum():,}")
  print(f"  Orders without reviews: {df['review_score'].isna().sum():,}")

  # Join payments (Left join - keep all orders)
  df = df.merge(payments_clean,how='left', on='order_id')
  print(f"After joining payments: {len(df):,} rows")

  # Final sanity check
  assert df['order_id'].duplicated().sum() == 0, f"PROBLEM: {df['order_id'].duplicated().sum()} duplicate order_id after join!"
  print(f"\nFinal master orders: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
  print("No duplicate order_ids")
  return df

master_orders = build_master_orders(orders_clean,orders_revenue,customers_clean,reviews_clean,payments_clean)




Start: 96,469 rows
After joining revenue: 96,469 rows (dropped 0 orders with no items)
After joining customers: 96,469 rows
After joining reviews: 96,469 rows
  Orders with reviews: 94,773
  Orders without reviews: 1,696
After joining payments: 96,469 rows

Final master orders: 96,469 rows x 24 columns
No duplicate order_ids


---

### 6.2 · `master_products` — Item-level grain

Built separately from `master_orders` because `order_items` has **multiple rows per order**.

> ⚠️ **Why not merge into `master_orders`?**  
> An order with 3 items would create 3 rows in `master_orders`, making every order-level  
> metric (revenue totals, customer counts, delivery days) triple-counted for that order.  
> Two separate dataframes at different granularities is the correct structure.

**One row = one item sold.**  
Used for: product category analysis · category revenue trends · items-per-order metrics

**Join sequence:**
```
items_clean
  ← products_clean      (LEFT — category_english per product_id)
  ← orders_clean        (LEFT — purchase_month/year/day/hour per order_id)
```


In [ ]:
master_products = (items_clean.
                   merge(products_clean,how='left', on='product_id').
                   merge(orders_clean[['order_id','purchase_month', 'purchase_year','purchase_day','purchase_hour']], how='left', on='order_id'))

# Check join quality
unmatched_products = master_products['category_english'].isna().sum()
unmatched_pct = unmatched_products/ len(master_products) * 100
print(f"Items with no matching product category: {unmatched_products:,} ({unmatched_pct:.2f}%)")
print(f"Items with no matching order date: {master_products['purchase_month'].isna().sum():,}")

assert len(master_products) == len(items_clean), f"Row count changed! items_clean: {len(items_clean):,}, master_products: {len(master_products):,}"

print(f"\n master_products: {master_products.shape}")

Items with no matching product category: 0 (0.00%)
Items with no matching order date: 2,462

 master_products: (112650, 13)


## 7 · Export Clean Data

Exporting all cleaned tables as CSVs to `/content/clean/`.

**Individual clean tables** → for Power BI (load separately, build relationships in Power BI)  
**Master dataframes** → for Python EDA notebook (load directly, analysis-ready)

> **Note:** `master_products` is exported as `msater_products.csv` (typo from development).  
> When re-running this notebook, correct the filename to `master_products.csv`.


In [ ]:
orders_clean.to_csv("/content/clean/orders.csv", index=False)
reviews_clean.to_csv("/content/clean/reviews.csv", index=False)
payments_clean.to_csv("/content/clean/payments.csv", index= False)
customers_clean.to_csv("/content/clean/customers.csv", index=False)
products_clean.to_csv("/content/clean/products.csv", index=False)
items_clean.to_csv("/content/clean/order_items.csv", index=False)
master_orders.to_csv("/content/clean/master_orders.csv", index= False)
master_products.to_csv('/content/clean/msater_products.csv', index=False)

print("All clean files exported ✓")
print("\nFor Power BI: load the individual clean tables and create relationships")
print("For Python analysis: use master_orders and master_products")

All clean files exported ✓

For Power BI: load the individual clean tables and create relationships
For Python analysis: use master_orders and master_products


In [ ]:
reviews_text_clean.to_csv("/content/clean/reviews_text_clean.csv", index=False)

## 8 · Cleaning Summary

### Final dataframe shapes

| Dataframe | Rows | Columns | Grain |
|-----------|------|---------|-------|
| `master_orders` | 96,469 | 24 | One row per delivered order |
| `master_products` | 112,650 | 13 | One row per item sold |

### Rows dropped per table

| Table | Raw | Clean | Dropped | Primary reason |
|-------|-----|-------|---------|---------------|
| `orders` | 99,441 | 96,469 | 2,972 | Non-delivered orders + null delivery dates |
| `order_items` | 112,650 | 112,650 | 0 | No issues found |
| `customers` | 99,441 | 99,441 | 0 | City standardisation only (no rows dropped) |
| `reviews` | 99,224 | 97,530 | 1,694 | Duplicate `review_id`s + duplicate order submissions |
| `products` | 32,951 | 32,951 | 0 | Null categories filled, not dropped |
| `payments` | 103,886 | aggregated | — | Aggregated to order level |

### Key decisions made

1. **Filter to delivered orders only** — revenue analysis requires complete transactions
2. **Keep two review versions** — `reviews_all` for scores, `reviews_text_clean` for sentiment
3. **Fill product category nulls with `'unknown'`** — preserves real sales data
4. **Use `customer_unique_id` not `customer_id`** — the only true person identifier
5. **Two separate analytical dataframes** — different granularities require separate tables

---

✅ **Pipeline complete.** Open `Olist_Sales_EDA.ipynb` to begin analysis.
